# Robust observation models：Student-tとContamination GP

観測値に外れ値が混入する場合の2種類のモデルを比較します。**Student-t**は裾の重い観測誤差、**Contamination mixture**は通常観測と汚染観測の混合を表現します。

Relevance Pursuitを用いた別の外れ値対策は [`13_robust_gp.ipynb`](13_robust_gp.ipynb) を参照してください。

## このNotebookで学ぶこと

- Student-t尤度とcontamination mixtureの**外れ値に対する仮定の違い**
- `num_inducing`、`df`、`contamination_probability` の役割
- モデルの `fit()` による専用損失の学習フローと、学習ステップ数による予測の変化
- 事後予測の可視化、有限候補集合からの候補提案
- Mixed入力・MultiTask入力への拡張

単一タスクの可視化では学習前・3ステップ後・十分に長く学習した後の予測を比較します。学習回数は収束や精度を保証しないため、真値とのRMSEも確認します。Mixed・MultiTaskの例にも独立した学習回数を設定します。

## 準備


In [ ]:
import torch
from robotorchan.models import StudentTSingleTaskGP, ContaminatedSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.linspace(0, 1, 16).unsqueeze(-1)
Y_true = torch.sin(2 * torch.pi * X)
Y = Y_true.clone()
outlier_index = 7
outlier_shift = 2.0
outlier_mask = torch.arange(X.shape[0], device=X.device) == outlier_index
clean_mask = ~outlier_mask
Y[outlier_mask] += outlier_shift

# Verify the synthetic observation process in original response units.
torch.testing.assert_close(Y[clean_mask], Y_true[clean_mask])
torch.testing.assert_close(
    Y[outlier_mask] - Y_true[outlier_mask],
    torch.full_like(Y[outlier_mask], outlier_shift),
)
assert outlier_mask.sum().item() == 1
print("Injected outlier:", X[outlier_index].item(),
      "true:", Y_true[outlier_index].item(),
      "observed:", Y[outlier_index].item())

# 比較用の学習回数。少数ステップは初期挙動の比較にだけ使用する。
initial_steps = 3
fit_steps = 400
extension_steps = 250


### 合成データの意味

真の関数は $f(x)=\sin(2\pi x)$ です。16点のうち1点（インデックス7）だけに +2.0 の外れ値を加えます。`Y_true` は汚染前の真値、`Y` はモデルに渡す観測値です。正常点は真値と一致し、汚染点は $x=7/15$ で真値約0.208、観測値約2.208です。

実データでは外れ値の場所や真の関数が分からないことが一般的です。ここではモデルの違いを理解するため、外れ値の位置を既知にしています。

## Student-t likelihood


In [ ]:
student = StudentTSingleTaskGP(X, Y, num_inducing=8, df=4.0)
comparison_X = torch.linspace(0, 1, 101).unsqueeze(-1)
with torch.no_grad():
    student.eval()
    student_initial = student.posterior(comparison_X).mean.detach().reshape(-1)
student.fit(steps=initial_steps, lr=0.03)
with torch.no_grad():
    student_short = student.posterior(comparison_X).mean.detach().reshape(-1)
student.fit(steps=fit_steps - initial_steps, lr=0.03)
print("supports_fit:", student.supports_fit)
print(student.posterior(torch.tensor([[0.25], [0.75]])).mean)


### Student-t likelihoodの考え方

ガウス分布より裾の重いStudent-t分布は、大きな残差が発生する可能性を比較的高く見積もります。`df=4.0` は自由度で、一般に自由度が小さいほど裾が重くなります。

`num_inducing=8` は近似推論で使う誘導点数に関する設定です。`fit()` は内部で既存の `training_loss()`（Student-t の期待対数尤度と変分KL項）をAdamで最小化します。通常のExact GPの `fit_gpytorch_mll()` とは学習経路が異なります。

## Contamination mixture


In [ ]:
contaminated = ContaminatedSingleTaskGP(
    X, Y, contamination_probability=0.1, num_inducing=16, inlier_scale=0.05, outlier_scale=2.0, num_likelihood_samples=64
)
with torch.no_grad():
    contaminated.eval()
    contaminated_initial = contaminated.posterior(comparison_X).mean.detach().reshape(-1)
contaminated.fit(steps=initial_steps, lr=0.03)
with torch.no_grad():
    contaminated_short = contaminated.posterior(comparison_X).mean.detach().reshape(-1)
contaminated.fit(steps=fit_steps - initial_steps, lr=0.02, warmup_steps=100)
print("supports_fit:", contaminated.supports_fit)
with torch.no_grad():
    print("final contaminated training loss:", contaminated.training_loss().item())
    print(contaminated.contamination_diagnostic(X, Y).squeeze(-1))


### Contamination mixtureの考え方

Contaminationモデルは、観測が通常の誤差過程から生じたか、汚染過程から生じたかを混合的に表現します。`contamination_probability=0.1` は汚染成分の混合確率の設定です。

`num_likelihood_samples=64` は既定のGauss–Hermite求積の点数（最低32点）を制御します。`fit()` は期待対数混合尤度とKLを含む負ELBOをAdamで最小化します。`warmup_steps=100` は通常成分の標準偏差を一時的に広げ、目標値0.05へ徐々に戻すことで、初期の正常点が汚染成分に吸収される問題を緩和します。通常成分の標準偏差は0.05、汚染成分は2.0とし、注入した外れ値の大きさに合わせています。これらは元の応答単位で指定する既知の設定であり、実データでは別途推定・検証が必要です。

## Posterior-based candidate proposals

Student-tとContaminated GPの事後平均を比較し、各モデルの候補を評価します。


In [ ]:
candidate_grid = torch.linspace(0, 1, 101).unsqueeze(-1)
for name, surrogate in [("Student-t", student), ("Contaminated", contaminated)]:
    surrogate.eval()
    with torch.no_grad():
        scores = surrogate.posterior(candidate_grid).mean.squeeze(-1)
    best_index = scores.argmax().item()
    candidate = candidate_grid[best_index:best_index + 1]
    candidate_Y = torch.sin(2 * torch.pi * candidate)
    assert candidate.shape == (1, 1)
    assert torch.isfinite(candidate_Y).all()
    print(name, "candidate:", candidate, "evaluated objective:", candidate_Y)


### 事後平均と真の関数を比較

真値、観測値、学習後の事後平均をすべて元の応答単位で表示します。モデルは出力を標準化していないため、予測に手動の逆変換を行いません。

さらに、学習前と3ステップ後の曲線を別パネルで比較し、学習不足による振幅の違いを確認します。学習後の曲線についても収束や頑健性を無条件に仮定せず、真値に対するRMSEと予測範囲を表示します。

正常観測点は青丸、汚染観測点は赤い ×、汚染位置の真値は白抜き丸で描き分けます。赤い点線は人工的に加えた +2.0 の差を表します。モデルはこのラベルを学習に利用せず、汚染点を含む全観測値 `Y` で学習します。

In [ ]:
import matplotlib.pyplot as plt

def plot_observations(ax):
    """Show clean and corrupted observations once, in original units."""
    ax.scatter(
        X[clean_mask, 0], Y[clean_mask, 0],
        s=22, color="tab:blue", label="Clean observations", zorder=4,
    )
    ax.scatter(
        X[outlier_mask, 0], Y[outlier_mask, 0],
        s=90, marker="x", color="tab:red", label="Corrupted observation", zorder=5,
    )
    ax.scatter(
        X[outlier_mask, 0], Y_true[outlier_mask, 0],
        s=44, facecolors="none", edgecolors="tab:red",
        label="Truth at corrupted x", zorder=5,
    )
    ax.vlines(
        X[outlier_mask, 0], Y_true[outlier_mask, 0], Y[outlier_mask, 0],
        colors="tab:red", linestyles=":", linewidth=1.2, alpha=0.8,
    )


plot_X = comparison_X
truth = torch.sin(2 * torch.pi * plot_X).reshape(-1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
curves = [
    ("Before fit", student_initial, contaminated_initial),
    (f"After {initial_steps} steps", student_short, contaminated_short),
]
with torch.no_grad():
    student.eval()
    contaminated.eval()
    student_mean = student.posterior(plot_X).mean.reshape(-1)
    contaminated_mean = contaminated.posterior(plot_X).mean.reshape(-1)
curves.append((f"After {fit_steps} steps", student_mean, contaminated_mean))

for ax, (title, student_pred, contaminated_pred) in zip(axes, curves, strict=True):
    ax.plot(plot_X[:, 0], truth, "--", color="0.35", label="True latent function")
    plot_observations(ax)
    for label, predicted in [("Student-t", student_pred), ("Contaminated", contaminated_pred)]:
        assert torch.isfinite(predicted).all()
        color = "tab:orange" if label == "Student-t" else "tab:green"
        ax.plot(plot_X[:, 0], predicted, color=color, label=label)
        rmse = (predicted - truth).square().mean().sqrt()
        print(f"{title}, {label}: RMSE={rmse.item():.4f}, "
              f"range=[{predicted.min().item():.3f}, {predicted.max().item():.3f}]")
    ax.set_title(title)
    ax.set_xlabel("x")
axes[0].set_ylabel("Response (original units)")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8)
fig.tight_layout(rect=(0, 0.13, 1, 1))
plt.show()

### 事後平均・不確実性・真値からの残差

Student-t と Contamination を別々のパネルで表示します。事後平均の周りの帯は **潜在関数の平均 ± 1.96 × 事後標準偏差**です。観測誤差を含む Student-t / 混合尤度の予測区間ではありません。

下段の残差は **予測平均 − 真値**を示します。合成例なので未観測点の真値を描けますが、実データでは通常わかりません。学習後の予測を表示しますが、単一の合成例であり、モデル間の一般的な優劣を判断する図ではありません。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex="col", sharey="row")
for col, (name, surrogate) in enumerate(
    [("Student-t", student), ("Contamination", contaminated)]
):
    with torch.no_grad():
        post = surrogate.posterior(plot_X)
        prediction = post.mean.squeeze(-1)
        std = post.variance.clamp_min(0).sqrt().squeeze(-1)
    true_values = torch.sin(2 * torch.pi * plot_X).squeeze(-1)
    ax = axes[0, col]
    ax.plot(plot_X[:, 0], true_values, "--", color="0.35", label="True latent")
    ax.plot(plot_X[:, 0], prediction, label="Posterior mean")
    ax.fill_between(
        plot_X[:, 0], prediction - 1.96 * std, prediction + 1.96 * std,
        alpha=0.2, label="Approx. 95% latent interval",
    )
    plot_observations(ax)
    ax.set_title(name)
    ax.set_ylabel("Response")
    ax.legend(fontsize=7)

    residual = prediction - true_values
    axes[1, col].plot(plot_X[:, 0], residual, label="Mean - truth")
    axes[1, col].axhline(0.0, color="gray", linestyle="--", linewidth=1)
    axes[1, col].set_xlabel("x")
    axes[1, col].set_ylabel("Residual")
    axes[1, col].legend(fontsize=8)
    print(f"{name} grid RMSE: {float(residual.square().mean().sqrt()):.4f}")
plt.tight_layout()
plt.show()

### 汚染診断スコアはどの観測点で高くなるか

`contamination_diagnostic(X, Y)` を観測点ごとに描画します。赤い線は**人工的に外れ値を入れた index 7** です。診断スコアは潜在平均を使った近似であり、確定的な外れ値ラベルでも厳密な事後汚染確率でもありません。学習後でもモデル仮定によって十分に判別できない場合があります。

In [ ]:
with torch.no_grad():
    contamination_scores = contaminated.contamination_diagnostic(X, Y).squeeze(-1)
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(torch.arange(len(X)), contamination_scores, label="Diagnostic score")
ax.axvline(outlier_index, linestyle="--", color="tab:red",
           label=f"Injected outlier: index {outlier_index}")
ax.set_xlabel("Observation index")
ax.set_ylabel("Contamination diagnostic")
ax.set_ylim(0.0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

## Mixed-input robust observation models

カテゴリ入力を含むStudent-tおよびContaminationモデルでも、学習、Posterior、有限プールからの候補選択と目的関数評価を行います。候補選択は獲得関数最適化ではありません。


### Contamination diagnosticの解釈

`contamination_diagnostic(X, Y)` は、モデルが観測値の汚染をどう評価するかを調べるための診断です。**出力をそのまま確定的な外れ値ラベルとして扱わないでください**。推定結果はモデル仮定や学習状況に依存します。

外れ値を除去すべきかどうかは、測定ミス・工程異常・本来の分布の裾など、発生原因を確認して判断します。

In [ ]:
from robotorchan.models import (
    MixedContaminatedSingleTaskGP,
    MixedStudentTSingleTaskGP,
)

mixed_X = torch.cat(
    [X, torch.arange(X.shape[0]).remainder(2).to(X).unsqueeze(-1)],
    dim=-1,
)
mixed_Y = Y + 0.2 * mixed_X[:, 1:2]
mixed_grid = torch.cartesian_prod(
    torch.linspace(0, 1, 31), torch.tensor([0.0, 1.0])
)
mixed_models = [
    (
        "Mixed Student-t",
        MixedStudentTSingleTaskGP(
            mixed_X, mixed_Y, cat_dims=[1], num_inducing=8, df=4.0
        ),
    ),
    (
        "Mixed contaminated",
        MixedContaminatedSingleTaskGP(
            mixed_X, mixed_Y, cat_dims=[1], num_inducing=8,
            contamination_probability=0.1, outlier_scale=2.0, num_likelihood_samples=32,
        ),
    ),
]
for name, model in mixed_models:
    if isinstance(model, ContaminatedSingleTaskGP):
        model.fit(steps=extension_steps, lr=0.02, warmup_steps=75)
    else:
        model.fit(steps=extension_steps, lr=0.03)
    with torch.no_grad():
        scores = model.posterior(mixed_grid).mean.reshape(-1)
    candidate = mixed_grid[scores.argmax()].unsqueeze(0)
    observation = (
        torch.sin(2 * torch.pi * candidate[:, :1])
        + 0.2 * candidate[:, 1:2]
    )
    assert torch.isfinite(scores).all()
    assert candidate[0, 1].item() in (0.0, 1.0)
    print(name, "candidate:", candidate, "evaluated objective:", observation)


### Mixed版：カテゴリごとの予測と提案候補

カテゴリ0と1を別の設計条件として扱い、連続変数 `x` を走査したときの予測平均と真値を比較します。星印は既存の**有限候補プール上で事後平均を最大化した候補**です。

合成関数のカテゴリ差は0.2ですが、学習回数を増やしても、その差が正しく再現されるとは限らないため、真値と同じ単位・共通の縦軸で確認します。カテゴリ値を連続値として補間する図ではありません。

In [ ]:
mixed_plot_x = torch.linspace(0.0, 1.0, 101)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, model) in zip(axes, mixed_models, strict=True):
    for category in (0, 1):
        design = torch.stack(
            [mixed_plot_x, torch.full_like(mixed_plot_x, float(category))],
            dim=-1,
        )
        with torch.no_grad():
            predicted = model.posterior(design).mean.squeeze(-1)
        truth = torch.sin(2 * torch.pi * mixed_plot_x) + 0.2 * category
        ax.plot(mixed_plot_x, predicted, label=f"Category {category}: mean")
        ax.plot(mixed_plot_x, truth, "--", alpha=0.5, label=f"Category {category}: truth")
    with torch.no_grad():
        pool_means = model.posterior(mixed_grid).mean.squeeze(-1)
    best = int(pool_means.argmax().item())
    selected = mixed_grid[best]
    ax.scatter(
        selected[0], pool_means[best], marker="*", s=140, zorder=5,
        label=f"Candidate (category {int(selected[1].item())})",
    )
    ax.set_title(name)
    ax.set_xlabel("Continuous variable x")
    ax.legend(fontsize=7)
axes[0].set_ylabel("Target response")
plt.tight_layout()
plt.show()

## Robust observation models with long-format multitask inputs

タスク列はモデル入力の最後の列に配置し、タスク0に固定した候補を予測平均で選択します。4モデルとも `fit()` を使います。内部ではそれぞれ既存の `training_loss()` を利用します。


### 候補提案の位置付け

既存コードでは101点の有限候補グリッドで**事後平均が最大の点**を選択しています。これはgreedyな活用であり、EI/UCBなどの不確実性を考慮する獲得関数最適化ではありません。

候補の評価は真の合成関数で行います。実際の観測では、新しい点にも外れ値が混入する可能性があります。

In [ ]:
from robotorchan.models import (
    ContaminatedMultiTaskGP,
    MixedContaminatedMultiTaskGP,
    MixedStudentTMultiTaskGP,
    StudentTMultiTaskGP,
)

task0_X = torch.cat([X, torch.zeros_like(X)], dim=-1)
task1_X = torch.cat([X, torch.ones_like(X)], dim=-1)
task_X = torch.cat([task0_X, task1_X], dim=0)
task_Y = torch.cat([Y, Y + 0.2], dim=0)
task_pool = torch.cat(
    [candidate_grid, torch.zeros_like(candidate_grid)], dim=-1
)
mixed_task0_X = torch.cat(
    [mixed_X, torch.zeros(mixed_X.shape[0], 1)], dim=-1
)
mixed_task1_X = torch.cat(
    [mixed_X, torch.ones(mixed_X.shape[0], 1)], dim=-1
)
mixed_task_X = torch.cat([mixed_task0_X, mixed_task1_X], dim=0)
mixed_task_Y = torch.cat([mixed_Y, mixed_Y + 0.2], dim=0)
mixed_task_pool = torch.cat(
    [mixed_grid, torch.zeros(mixed_grid.shape[0], 1)], dim=-1
)
robust_task_models = [
    (
        "Student-t multitask",
        StudentTMultiTaskGP(task_X, task_Y, task_feature=1, num_inducing=8),
        task_pool,
    ),
    (
        "Contaminated multitask",
        ContaminatedMultiTaskGP(
            task_X, task_Y, task_feature=1, num_inducing=8,
            outlier_scale=2.0, num_likelihood_samples=32,
        ),
        task_pool,
    ),
    (
        "Mixed Student-t multitask",
        MixedStudentTMultiTaskGP(
            mixed_task_X, mixed_task_Y, task_feature=2,
            cat_dims=[1], num_inducing=8,
        ),
        mixed_task_pool,
    ),
    (
        "Mixed contaminated multitask",
        MixedContaminatedMultiTaskGP(
            mixed_task_X, mixed_task_Y, task_feature=2,
            cat_dims=[1], num_inducing=8, outlier_scale=2.0, num_likelihood_samples=32,
        ),
        mixed_task_pool,
    ),
]
for name, model, pool in robust_task_models:
    if isinstance(model, ContaminatedSingleTaskGP):
        model.fit(steps=extension_steps, lr=0.02, warmup_steps=75)
    else:
        model.fit(steps=extension_steps, lr=0.03)
    with torch.no_grad():
        scores = model.posterior(pool).mean.reshape(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    objective = torch.sin(2 * torch.pi * candidate[:, :1])
    if candidate.shape[-1] == 3:
        objective = objective + 0.2 * candidate[:, 1:2]
    assert torch.isfinite(scores).all()
    assert torch.isfinite(objective).all()
    print(name, "candidate:", candidate, "evaluated objective:", objective)


### MultiTask版：タスク間の応答の違いを確認

4モデルで **task 0 と task 1 の事後平均**を比較します。Mixed版のカテゴリ値は0に固定し、共通の連続変数 `x` だけを動かします。破線は合成例の真値で、task 1 の応答は task 0 より0.2高い設定です。

ここでの task ID はカテゴリ設計変数と異なり、**どの応答を予測するか**を指定します。共通の縦軸で曲線を比較し、真値からのずれを確認します。

In [ ]:
mt_plot_x = torch.linspace(0.0, 1.0, 101)
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, (name, model, _) in zip(axes.flat, robust_task_models, strict=True):
    is_mixed = name.startswith("Mixed")
    for task_id in (0, 1):
        columns = [mt_plot_x]
        if is_mixed:
            columns.append(torch.zeros_like(mt_plot_x))  # Category 0.
        columns.append(torch.full_like(mt_plot_x, float(task_id)))
        query_X = torch.stack(columns, dim=-1)
        with torch.no_grad():
            predicted = model.posterior(query_X).mean.squeeze(-1)
        truth = torch.sin(2 * torch.pi * mt_plot_x) + 0.2 * task_id
        ax.plot(mt_plot_x, predicted, label=f"Task {task_id}: mean")
        ax.plot(mt_plot_x, truth, "--", alpha=0.5, label=f"Task {task_id}: truth")
    ax.set_title(name)
    ax.set_xlabel("x (category 0 for mixed)")
    ax.set_ylabel("Response")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### MultiTask入力への拡張

MultiTaskモデルでは最後の列をtask IDとして使い、複数タスクの観測をlong-formatで学習します。task IDは**どのタスクの応答を予測するか**を表し、Mixedモデルのカテゴリ設計変数とは役割が異なります。

末尾のコードではtask 0に固定した候補集合を使い、Student-t・Contaminationそれぞれの通常版とMixed版を比較します。

### Mixed入力への拡張

カテゴリ設計変数を追加したMixedモデルでは、連続変数とカテゴリ値0/1の組合せを候補集合として列挙します。`cat_dims=[1]` は2列目がカテゴリであることを示します。

カテゴリを0.5などの連続値として補間しません。ここでもモデル学習、posterior、候補選択、評価を既存APIで行います。

## まとめ：どの外れ値モデルを選ぶか

| 状況 | 検討するモデル | 留意点 |
|---|---|---|
| 大きな残差が連続的に起こり得る | Student-t | 自由度や推定安定性を確認 |
| 通常観測と汚染観測が混在する | Contamination | 混合確率・診断結果を過信しない |
| 少数の大きな観測異常を明示的に扱いたい | Relevance Pursuit（Notebook 13） | 別のモデル化仮定 |

いずれも**観測モデルの頑健性**を扱います。入力摂動に頑健な目的関数やリスク指標に基づく最適化とは区別してください。実運用では学習収束、予測精度、校正、外れ値率への感度を比較して選択します。